In [1]:
import pandas as pd
import numpy as np

pd.read_csv("../data/processed/kontext_monat.csv").columns.tolist()

['jahr',
 'monat',
 'eur_chf',
 'ferientage_de',
 'ferientage_ch',
 'ferientage_ch_annahme',
 'feiertage_ch']

In [2]:
panel = pd.read_csv("../data/processed/panel_winter.csv")
panel = panel.sort_values(["bfs_nr", "jahr", "monat"])
panel["schnee_saison"] = panel.groupby(["bfs_nr", "winter"])["schneefall_cm"].cumsum()
print(panel.shape, panel["bfs_nr"].nunique())      # erwartet: 2255 Zeilen, 45 Orte

(2255, 31) 45


Abweichungen statt Rohwerte: Wir messen, ob **derselbe Ort** bei mehr Schnee voller ist als sonst. Die Normalwerte stammen nur aus den Trainingswintern 2013/14–2021/22.

In [3]:
m = panel[panel.covid == 0].copy()                  # Corona-Monate entfernen

train_mask = m["winter"] <= 2021
norm = (m[train_mask].groupby(["bfs_nr", "monat"])
        [["zimmerauslastung", "schneefall_cm", "temp_mittel", "schnee_saison"]].mean()
        .add_suffix("_norm").reset_index())
m = m.merge(norm, on=["bfs_nr", "monat"], how="left")

m["y_abw"]             = m["zimmerauslastung"] - m["zimmerauslastung_norm"]   # Prozentpunkte
m["schnee_abw"]        = m["schneefall_cm"]    - m["schneefall_cm_norm"]      # cm
m["temp_abw"]          = m["temp_mittel"]      - m["temp_mittel_norm"]        # °C
m["schnee_saison_abw"] = m["schnee_saison"]    - m["schnee_saison_norm"]
print(len(m))                                        # erwartet: 2032


2032


In [4]:
m = m.sort_values(["bfs_nr", "jahr", "monat"])

# Kontext anhängen, falls nicht schon in den Daten
if "eur_chf" not in m.columns:
    kontext = pd.read_csv("../data/processed/kontext_monat.csv")
    n0 = len(m)
    m = m.merge(kontext, on=["jahr", "monat"], how="left", validate="m:1")
    assert len(m) == n0

m["schnee_vormonat"] = m.groupby(["bfs_nr", "winter"])["schneefall_cm"].shift(1)
if "sonnenstunden" in m.columns:                      # nur, falls nachgeladen
    m["sonne_abw"] = m["sonnenstunden"] - m.groupby(["bfs_nr", "monat"])["sonnenstunden"].transform("mean")

m["log_betten"]        = np.log(m["betten"])
m["anteil_ausland_vj"] = m.groupby(["bfs_nr", "monat"])["anteil_ausland"].shift(1)   # Vorjahr, kein Leakage
m["trend"]             = m["winter"] - 2013                                           # 0 = Winter 2013/14


Zeitlicher Split statt Zufall, damit das Modell nichts aus der Zukunft lernt.

In [5]:
train = m[m["winter"] <= 2021]      # Winter 2013/14 – 2021/22
test  = m[m["winter"] >= 2022]      # Winter 2022/23 – 2025/26
print(len(train), len(test), len(train) + len(test))    # Test ca. 720, zusammen 2032

1312 720 2032


In [6]:
print(train["winter"].max(), test["winter"].min())       # 2021 und 2022
print(train["covid"].sum() + test["covid"].sum())        # 0

neu = ["y_abw", "schnee_abw", "temp_abw", "schnee_saison_abw", "schnee_vormonat",
       "eur_chf", "ferientage_de", "ferientage_ch", "feiertage_ch",
       "log_betten", "anteil_ausland_vj", "trend"]
print(m[neu].isna().sum())
print(train[["y_abw", "schnee_abw", "temp_abw"]].describe())


2021 2022
0
y_abw                  0
schnee_abw             0
temp_abw               0
schnee_saison_abw      0
schnee_vormonat      520
eur_chf                0
ferientage_de          0
ferientage_ch          0
feiertage_ch           0
log_betten             0
anteil_ausland_vj    180
trend                  0
dtype: int64
              y_abw    schnee_abw      temp_abw
count  1.312000e+03  1.312000e+03  1.312000e+03
mean   7.040439e-17  3.899320e-16  1.353931e-17
std    5.797581e+00  2.410035e+01  1.829455e+00
min   -2.550000e+01 -6.656125e+01 -4.705645e+00
25%   -3.750000e+00 -1.538687e+01 -1.427722e+00
50%    0.000000e+00 -1.733125e+00  3.572581e-01
75%    3.428571e+00  1.332844e+01  1.275605e+00
max    2.185714e+01  1.504388e+02  5.593145e+00


In [7]:
print("ferientage_ch_annahme" in m.columns)  

True
